## 第 5 课：块内归约 tl.sum

题目：[Triton: Row-wise Sum](https://www.deep-ml.com/problems/971?from=Triton%20Essentials)（ID 971）

计算目标：

In [ ]:
output[m] = sum_n x[m, n]

x 的形状是 `(M, N)`，output 是长度为 `M` 的一维 Tensor，每个元素是一整行的和。

例如：

In [ ]:
x = [[1, 2, 3],
     [4, 5, 6]]

output = [6, 15]

前几课是"一个 program 管一块连续元素，逐元素运算"。这一课第一次出现**归约**：多个元素要合并成一个标量。

### 1. 一个 program 负责一行

这次 grid 是 1-D 的，但每个 program 管一整行：

In [ ]:
pid = tl.program_id(0)  # 第几行
grid = (M,)

pid 既是行号，也是输出下标。`M` 行 → `M` 个 program → `M` 个输出元素。

### 2. 整行装入寄存器

一行有 N 个元素，要在一个 program 里装下，块大小必须 >= N：

In [ ]:
BLOCK_SIZE_N = triton.next_power_of_2(N)  # 题目保证 N <= 16384
offs_n = tl.arange(0, BLOCK_SIZE_N)
mask = offs_n < N

`tl.arange` 要求长度是 2 的幂，所以用 `next_power_of_2(N)` 向上取整。

### 3. 加载时给 padding 传 other=0.0

块比真实行宽（比如 N=100, BLOCK_SIZE_N=128），多出来的 28 个 lane 是 padding。求和时它们必须贡献 0：

In [ ]:
row = tl.load(
    x_ptr + pid * stride_xm + offs_n,
    mask=mask,
    other=0.0,
)

### 4. tl.sum 归约 + 标量存储

In [ ]:
row_sum = tl.sum(row, axis=0)  # 1-D block → 标量
tl.store(output_ptr + pid, row_sum)

输出只有一个数，下标直接用 pid。

## 你的代码骨架

In [ ]:
import torch
import triton
import triton.language as tl


@triton.jit
def row_sum_kernel(
    x_ptr,
    output_ptr,
    M,
    N,
    stride_xm,
    BLOCK_SIZE_N: tl.constexpr,
):
    # TODO 1：取得行 ID（这个 ID 就是行号，也是输出下标）

    # TODO 2：生成列下标 offs_n（0 ～ BLOCK_SIZE_N-1）

    # TODO 3：生成 mask（offs_n < N）

    # TODO 4：加载整行
    # mask 之外的 lane 用 other=0.0，不参与求和

    # TODO 5：tl.sum 归约到标量

    # TODO 6：把标量写入 output_ptr + pid
    pass


def row_sum(x: torch.Tensor) -> torch.Tensor:
    # TODO 7：取得 M、N

    # TODO 8：BLOCK_SIZE_N = triton.next_power_of_2(N)

    # TODO 9：分配 output（形状 (M,)）

    # TODO 10：创建一维 grid（M,）

    # TODO 11：启动 kernel

    # TODO 12：返回 output
    pass

同时回答：

1. `M=4, N=100` 时，BLOCK_SIZE_N 是多少？grid 是多少？每个 program 实际处理多少个元素（含 padding）？
2. 为什么加载时要给 mask 外的 lane 显式传 `other=0.0`？如果不传会怎样？（提示：这些 lane 的值会参与 `tl.sum`）
3. 为什么 `BLOCK_SIZE_N` 必须是 2 的幂、并且 >= N？`tl.arange` 有什么限制？

把代码和三个答案发给我，我继续审查。